In [ ]:
#
# Import Landsat Data and Calculate NDVI
#

import rasterio
import os
import geopandas as gpd
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import gdown
from scipy.stats import linregress
import cmcrameri.cm as cmc
from matplotlib.colors import LightSource

# Loop over years to create filepaths filepaths
years = [1985, 1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]

ls_paths = []

for year in years:
	fp = "data/raw/LandsatComposite_Zurich_" + str(year) + ".tif"

	ls_paths.append(fp)

# Loop over filepaths to import data into data cube
list_ds = []

for path in ls_paths:
	da = rioxarray.open_rasterio(path)

	dt = int(path.split("_")[2].split(".")[0])

	ds = xr.Dataset(
		{
			"Blue": da.sel(band = 1).drop_vars("band", errors="ignore"), 
			"Green": da.sel(band = 2).drop_vars("band", errors="ignore"), 
			"Red": da.sel(band = 3).drop_vars("band", errors="ignore"), 
			"NIR": da.sel(band = 4).drop_vars("band", errors="ignore"), 
			"SWIR1":  da.sel(band = 5).drop_vars("band", errors="ignore"), 
			"SWIR2": da.sel(band = 6).drop_vars("band", errors="ignore"), 
			"TIR": da.sel(band = 7).drop_vars("band", errors="ignore"),
		}
	)

	#ds = ds.assign_coords(band = bands)
	ds = ds.expand_dims(time = [dt])
	ds["NDVI"] = (ds.NIR - ds.Red) / (ds.NIR + ds.Red)
	ds["LST"] = ds.TIR - 273.15
	
	list_ds.append(ds)

ls_cube = xr.combine_by_coords(list_ds)

#ls_cube

In [ ]:
#
# Visualise NDVI
#

ndvi_fg = ls_cube.NDVI.isel(time=slice(0,1)).plot(
	col="time", col_wrap=1, robust=True, cbar_kwargs={"label": "NDVI"}
)

ndvi_fg.map_dataarray(
	xr.plot.contour,
	x="x",
	y="y",
	colors="k",
	levels=5,
	linewidths=0.5,
	add_colorbar=False,
)
# Turn into subplot to turn off axes
plt.axis("off")
plt.show()

In [ ]:
#
# Difference 2024 to 1985
#

# LST Difference

lst_1985 = ls_cube.LST.sel(time=1985)
lst_2024 = ls_cube.LST.sel(time=2024)

lst_diff = lst_2024 - lst_1985

lst_diff_fg = lst_diff.plot(
	col_wrap=1, robust=True, cbar_kwargs={"label": "Temperature Difference (°C)"}
)

ndvi_fg.map_dataarray(
	xr.plot.contour,
	x="x",
	y="y",
	colors="k",
	levels=5,
	linewidths=0.5,
	add_colorbar=False,
)
# Turn into subplot to turn off axes
plt.title('Temperature Difference (1985-2024)')
plt.axis("off")
plt.show()

# NDVI Difference

ndvi_1985 = ls_cube.NDVI.sel(time=1985)
ndvi_2024 = ls_cube.NDVI.sel(time=2024)

ndvi_diff = ndvi_2024 - ndvi_1985

ndvi_diff_fg = ndvi_diff.plot(
	col_wrap=1, robust=True, cbar_kwargs={"label": "NDVI Difference"}
)

ndvi_fg.map_dataarray(
	xr.plot.contour,
	x="x",
	y="y",
	colors="k",
	levels=5,
	linewidths=0.5,
	add_colorbar=False,
)
# Turn into subplot to turn off axes
plt.title('Vegetation Difference (1985-2024)')
plt.axis("off")
plt.show()

In [ ]:
#
# Decadal Trends
#

# Temperature

# 1. Apply linear regression across the time dimension for the entire cube
trends = ls_cube.polyfit(dim="time", deg=1)

# 2. Extract and scale slope (degree=1 is change per year, * 10 for change per decade)
trend_map = trends.LST_polyfit_coefficients.sel(degree=1) * 10

# 4. Plot the regional trend map
fig, ax = plt.subplots()

trend_map.plot(
	ax=ax, 
	cmap='RdBu_r', 
	cbar_kwargs={'label': 'Temperature Change per Decade (°C)'}
)

ax.set_title("Decadal Temperature Trend (1985-2024)")
plt.axis("off")
plt.tight_layout()
plt.show()

# NDVI

# Apply linear regression
trends = ls_cube.polyfit(dim="time", deg=1)

trend_map = trends.NDVI_polyfit_coefficients.sel(degree=1) * 10

# Plot the regional trend map
fig, ax = plt.subplots()

trend_map.plot(
	ax=ax, 
	cmap='RdBu', 
	cbar_kwargs={'label': 'NDVI Change per Decade'}
)

ax.set_title("Decadal Trend of NDVI")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#
# Visualise 3-yearly Mean
#

ls_mean = ls_cube.mean(dim=["x", "y"])

# Plot mean LST

ls_mean.LST.sel().plot(marker="o", figsize=(10, 4))
plt.title("3-yearly mean Temperature (1985-2024)")
plt.ylabel("Temperature (°C)")
plt.xlabel("Year")
plt.xticks(years)
plt.grid(alpha=0.3)
plt.show()

# Plot mean NDVI
ls_mean.NDVI.sel().plot(marker="o", figsize=(10, 4))
plt.title("3-yearly mean NDVI (1985-2024)")
plt.ylabel("NDVI")
plt.xlabel("Year")
plt.xticks(years)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
#
# Visualise Correlation of NDVI and LST
#

xr.plot.scatter(ls_cube, x="NDVI", y="LST", xlim=[0, 1])
#xr.plot.scatter(ls_cube.sel(time=2024), x="NDVI", y="LST", xlim=[-1, 2])